# UIT DSC 2026 – LegalQA Step-by-Step Diagnostic & Smoke Test
### Quy trình kiểm thử dò từ dưới lên (Bottom-Up Verification) cho Kaggle / Colab / Local

Notebook này dùng để **kiểm tra lần lượt từng tầng** của hệ thống LegalQA RAG theo thứ tự từ dưới lên và **dừng ngay tại tầng đầu tiên bị lỗi** để cô lập nguyên nhân, thay vì chạy toàn bộ pipeline rồi mới debug ngược.

---

### Thứ tự 10 tầng kiểm tra:
| Tầng | Thành phần kiểm thử | Model được bật | Mục tiêu xác nhận |
| :--- | :--- | :--- | :--- |
| **1** | **Unit test & Môi trường** | Không model | Codebase đồng bộ, commit chuẩn, 58/58 unit tests pass |
| **2** | **Dữ liệu & Chunking** | Không model | Train 7.000 mẫu, Public Test hợp lệ, chunk 620/100 từ |
| **3** | **BM25 độc lập** | BM25 SQLite FTS5 | Build SQLite index, đo Recall@K BM25 độc lập (`--reranker-model ""`) |
| **4** | **Dense index độc lập** | Vietnamese_Embedding_v2 | Sanity encode (shape, norm, NaN), build Dense FAISS/NumPy |
| **5** | **BM25 vs Dense vs RRF** | BM25 + Dense (No Reranker) | So sánh 3 nhánh, xác nhận RRF Recall@5 > BM25 Recall@5 |
| **6** | **Reranker độc lập** | BM25 + Dense + Reranker | 4 stages (`bm25`, `dense`, `rrf`, `reranker`), kiểm tra thứ hạng |
| **7** | **Generator độc lập** | Vi-Qwen2-1.5B-RAG | Sinh câu trả lời từ context chuẩn, test max_tokens 512/1024 |
| **8** | **Full RAG (20 mẫu Train)** | Toàn pipeline RAG | Chạy `ALLOW_RETRIEVAL_FALLBACK=False`, log chi tiết từng bước |
| **9** | **Submission & Resume** | Public Test mini | Schema Codabench `{"id": {"answer": "..."}}`, test resume |
| **10** | **Kaggle Persistence** | Lưu trữ lâu dài | Kiểm tra file artifacts, model snapshot sẵn sàng cho Save Version |

> **Quy tắc vàng khi debug RAG:**
> - `mode="rag"` mới là pipeline BM25 + Dense + RRF + Reranker + Generator.
> - Luôn tắt `--allow-retrieval-fallback` khi test để lỗi không bị che giấu.
> - Khi đo BM25/Dense sạch, luôn truyền `--reranker-model ""`.


## 0. Thiết lập Môi trường, Cài đặt Dependencies & Khởi tạo


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import sqlite3
import subprocess
import sys
import time
import zipfile
from pathlib import Path

# Cấu hình PyTorch và Tokenizers tránh phân mảnh VRAM
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Dictionary lưu trạng thái pass/fail thực tế của từng tầng
TEST_STAGE_RESULTS: dict[str, bool] = {}

# 1. Nhận diện nền tảng thực thi
if Path("/kaggle/working").is_dir():
    RUNTIME_PLATFORM = "Kaggle"
    REPO_DIR = Path("/kaggle/working/uit-dsc-2026-task2-legalqa")
    INPUT_ROOT = Path("/kaggle/input")
    WORK_DIR = Path("/kaggle/working/legalqa-test-run")
elif Path("/content").is_dir():
    RUNTIME_PLATFORM = "Colab"
    REPO_DIR = Path("/content/uit-dsc-2026-task2-legalqa")
    INPUT_ROOT = Path("/content")
    WORK_DIR = Path("/content/legalqa-test-run")
else:
    RUNTIME_PLATFORM = "Local"
    REPO_DIR = Path(".").resolve()
    INPUT_ROOT = Path(".").resolve()
    WORK_DIR = REPO_DIR / "artifacts_test"

EXPORT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else WORK_DIR
WORK_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = WORK_DIR / "hf-cache"
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")

# 2. Cấu hình Model IDs
EMBEDDING_MODEL_ID = "AITeamVN/Vietnamese_Embedding_v2"
RERANKER_MODEL_ID = "AITeamVN/Vietnamese_Reranker"
GENERATOR_MODEL_ID = "AITeamVN/Vi-Qwen2-1.5B-RAG"

REPO_URL = "https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git"

def run(command: list[str], cwd: Path | None = None) -> int:
    """Chạy lệnh shell và in log trực tiếp."""
    cmd_str = " ".join(map(str, command))
    print(f"\n[RUN] {cmd_str}")
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        encoding="utf-8",
        errors="replace",
    )
    if process.stdout is not None:
        for line in iter(process.stdout.readline, ""):
            print(line, end="", flush=True)
        process.stdout.close()
    return process.wait()

print(f" Nền tảng: {RUNTIME_PLATFORM}")
print(f" Python: {sys.version.split()[0]}")
print(f" Repo Dir: {REPO_DIR}")
print(f" Work Dir: {WORK_DIR}")
print(f" Export Dir: {EXPORT_DIR}")

# 3. Clone hoặc pull repo nếu ở Kaggle/Colab
if RUNTIME_PLATFORM in ("Kaggle", "Colab"):
    if not (REPO_DIR / ".git").is_dir():
        run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    else:
        run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# 4. Tự động kiểm tra và cài đặt dependencies cho môi trường sạch
print("\n📦 Kiểm tra và cài đặt thư viện phụ thuộc...")
packages_to_check = [
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("tqdm", "tqdm"),
    ("nltk", "nltk"),
    ("rouge_score", "rouge-score"),
]
missing_packages = []
for mod_name, pip_name in packages_to_check:
    try:
        __import__(mod_name)
    except ImportError:
        missing_packages.append(pip_name)

if missing_packages:
    print(f"   Đang cài đặt các thư viện còn thiếu: {missing_packages}...")
    run([sys.executable, "-m", "pip", "install", "-q"] + missing_packages)

try:
    import nltk
    for res in ["wordnet", "punkt", "punkt_tab", "omw-1.4"]:
        nltk.download(res, quiet=True)
except Exception as exc:
    print(f"   NLTK data notice: {exc}")

# 5. Kiểm tra phần cứng GPU
import torch
print(f"\n PyTorch: {torch.__version__}")
print(f" CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" Số GPU: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB)")
    DEVICE = "cuda"
else:
    print("  Không phát hiện CUDA GPU, chạy CPU mode.")
    DEVICE = "cpu"


---
## Tầng 1. Unit Tests & Môi trường (Không bật Model)

Chạy bộ unit test tích hợp sẵn để kiểm tra tính toàn vẹn của logic toán học, text tokenizer, pipeline routing, KNN similarity, RRF, và CLI schema validation.


In [ ]:
# Kiểm tra commit git hiện tại
import subprocess

res = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True)
commit_hash = res.stdout.strip()
print(f" Mã nguồn Git commit: {commit_hash}")

# Chạy toàn bộ unit tests
print("\n--- BẮT ĐẦU CHẠY UNIT TESTS ---")
exit_code = run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=REPO_DIR)

TEST_STAGE_RESULTS["Tầng 1 (Unit Tests)"] = (exit_code == 0)

if exit_code == 0:
    print("\n [TẦNG 1 PASS] Tất cả unit tests đều pass sạch sẽ!")
else:
    print(f"\n [TẦNG 1 FAIL] Có lỗi trong unit tests (Exit code: {exit_code}). Hãy kiểm tra log phía trên trước khi tiếp tục!")
    raise RuntimeError("Tầng 1 thất bại: Unit test không pass.")


---
## Tầng 2. Dữ liệu & Chunking (Không bật Model)

Kiểm tra:
1. `train.json`: Đúng 7.000 mẫu, không bị trùng ID, không có `question` hay `answer` rỗng.
2. `public-official.json` / `public_test.json`: Kiểm tra định dạng schema test đầu vào.
3. Thư mục `selected-contexts`: Định vị đúng file `.json` hoặc file `.zip`, trích xuất thông tin `id`, `passage`, `name`.
4. Cấu hình chunking đồng bộ: `MAX_CHUNK_WORDS = 620`, `OVERLAP_WORDS = 100`.


In [ ]:
# 1. Tìm file dữ liệu
def find_dataset_files(root: Path) -> tuple[Path | None, Path | None, Path | None]:
    train_file = None
    public_file = None
    contexts_target = None
    
    for p in root.rglob("train.json"):
        if p.is_file():
            train_file = p
            break
            
    for name in ["public-official.json", "public_test.json", "test.json"]:
        for p in root.rglob(name):
            if p.is_file():
                public_file = p
                break
        if public_file:
            break
            
    for p in root.rglob("selected-contexts"):
        if p.is_dir():
            # Tránh thư mục cha nếu có lồng selected-contexts/selected-contexts
            sub = p / "selected-contexts"
            contexts_target = sub if sub.is_dir() else p
            break
    if not contexts_target:
        for p in root.rglob("selected-contexts.zip"):
            if p.is_file():
                contexts_target = p
                break

    return train_file, public_file, contexts_target

TRAIN_PATH, PUBLIC_PATH, CONTEXTS_PATH = find_dataset_files(INPUT_ROOT)
if not TRAIN_PATH and (REPO_DIR / "data" / "train.json").is_file():
    TRAIN_PATH = REPO_DIR / "data" / "train.json"
if not PUBLIC_PATH and (REPO_DIR / "data" / "public-official.json").is_file():
    PUBLIC_PATH = REPO_DIR / "data" / "public-official.json"
if not CONTEXTS_PATH and (REPO_DIR / "data" / "selected-contexts").is_dir():
    CONTEXTS_PATH = REPO_DIR / "data" / "selected-contexts"
if not CONTEXTS_PATH and (REPO_DIR / "selected-contexts.zip").is_file():
    CONTEXTS_PATH = REPO_DIR / "selected-contexts.zip"

print(f" Train path: {TRAIN_PATH}")
print(f" Public test path: {PUBLIC_PATH}")
print(f" Contexts path: {CONTEXTS_PATH}")

assert TRAIN_PATH and TRAIN_PATH.is_file(), "Không tìm thấy train.json!"
assert CONTEXTS_PATH and (CONTEXTS_PATH.is_dir() or CONTEXTS_PATH.is_file()), "Không tìm thấy selected-contexts!"

# 2. Kiểm tra chặt chẽ train.json
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

print(f" Tổng số mẫu Train: {len(train_data):,}")
assert len(train_data) > 0, "train.json rỗng!"

empty_q = 0
empty_a = 0
unique_ids = set()
for sample_id, item in train_data.items():
    unique_ids.add(str(sample_id))
    q = str(item.get("question", "")).strip()
    a = str(item.get("answer", "")).strip()
    if not q:
        empty_q += 1
    if not a:
        empty_a += 1

print(f" Unique IDs: {len(unique_ids):,}")
print(f" Câu hỏi rỗng: {empty_q}")
print(f" Đáp án rỗng: {empty_a}")

assert len(unique_ids) == len(train_data), "Có ID bị trùng trong train.json!"
assert empty_q == 0, f"Có {empty_q} câu hỏi rỗng trong train.json!"
assert empty_a == 0, f"Có {empty_a} đáp án rỗng trong train.json!"

# 3. Kiểm tra Public Test nếu có
if PUBLIC_PATH and PUBLIC_PATH.is_file():
    with open(PUBLIC_PATH, "r", encoding="utf-8") as f:
        public_data = json.load(f)
    print(f" Tổng số mẫu Public Test: {len(public_data):,}")
    assert isinstance(public_data, dict), "Public Test JSON phải là dictionary!"
    assert len(public_data) > 0, "Public Test JSON không được rỗng!"
    for p_id, p_item in list(public_data.items())[:10]:
        assert isinstance(p_item, dict), f"Mẫu {p_id} không phải dict!"
        assert isinstance(p_item.get("question"), str) and p_item.get("question").strip(), f"Mẫu {p_id} thiếu câu hỏi hợp lệ!"

# 4. Kiểm tra chunking logic
from legalqa_baseline.text import chunk_passage, tokenize
from legalqa_baseline.storage import iter_contexts

sample_text = """Văn bản: Luật Thử Nghiệm Số 01/2026/QH15.
Điều 1. Phạm vi điều chỉnh và đối tượng áp dụng
1. Luật này quy định chi tiết về các nguyên tắc, quy trình và phương pháp thử nghiệm hệ thống LegalQA tự động phục vụ cuộc thi UIT Data Science Challenge 2026.
2. Quy định này áp dụng đối với mọi tổ chức, cá nhân, thí sinh tham gia nghiên cứu và phát triển giải pháp trí tuệ nhân tạo trong lĩnh vực pháp luật Việt Nam."""

chunks = chunk_passage(sample_text, max_words=620, overlap_words=100)
print(f" Số chunk tạo ra từ văn bản mẫu: {len(chunks)}")
for idx, chunk_text in enumerate(chunks, start=1):
    print(f"   - [Chunk {idx}] ({len(tokenize(chunk_text))} tokens): {chunk_text[:60]}...")

assert len(chunks) >= 1, "Chunking không sinh ra chunk nào!"

# 5. Kiểm tra đọc contexts thực tế từ nguồn
print("\n Kiểm tra đọc context thực tế:")
sample_ctx = None
for i, ctx in enumerate(iter_contexts(CONTEXTS_PATH), start=1):
    if sample_ctx is None:
        sample_ctx = ctx
    if i >= 3:
        break

ctx_id = sample_ctx.get("id") or sample_ctx.get("context_id")
ctx_name = sample_ctx.get("name")
ctx_text = sample_ctx.get("passage") or sample_ctx.get("text") or ""

print(f"   - Sample Context ID: {ctx_id}")
print(f"   - Sample Context Name: {ctx_name}")
print(f"   - Sample Context Passage Length: {len(ctx_text):,} ký tự")
assert sample_ctx and ctx_text, "Không đọc được context thật từ đường dẫn!"

TEST_STAGE_RESULTS["Tầng 2 (Dữ liệu & Chunking)"] = True
print("\n [TẦNG 2 PASS] Dữ liệu và chunking hoàn toàn hợp lệ!")


---
## Tầng 3. BM25 Index & Retrieval Độc Lập (BM25 Only)

Xây dựng SQLite FTS5 index và đo lường độ phủ truy xuất (pseudo Recall@K) của BM25 độc lập, không bật Dense và không bật Reranker.


In [ ]:
DB_PATH = WORK_DIR / "legalqa_test.sqlite"
EVAL_BM25_PATH = WORK_DIR / "eval_bm25_test.json"

# 1. Build BM25 Index (hỗ trợ rerun với --force)
print("--- 3.1 XÂY DỰNG CHỈ MỤC BM25 SQLITE ---")
build_cmd = [
    sys.executable, "-m", "legalqa_baseline", "build-index",
    "--contexts", str(CONTEXTS_PATH),
    "--train", str(TRAIN_PATH),
    "--db", str(DB_PATH),
    "--max-chunk-words", "620",
    "--overlap-words", "100",
    "--force"
]
ret = run(build_cmd, cwd=REPO_DIR)
assert ret == 0, "Build BM25 Index thất bại!"

# 2. Kiểm tra bảng ảo contexts_fts và train_fts trong SQLite
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM contexts_fts")
    chunk_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM train_fts")
    train_count = cur.fetchone()[0]
    print(f" SQLite contexts_fts chunks: {chunk_count:,} dòng")
    print(f" SQLite train_fts QA pairs: {train_count:,} dòng")

assert chunk_count > 0, "Bảng contexts_fts trong SQLite rỗng!"

# 3. Đo lường retrieval Recall của BM25 độc lập (reranker-model="")
print("\n--- 3.2 ĐO RETRIEVAL BM25 ĐỘC LẬP (100 MẪU) ---")
eval_bm25_cmd = [
    sys.executable, "-m", "legalqa_baseline", "evaluate-retrieval",
    "--train", str(TRAIN_PATH),
    "--db", str(DB_PATH),
    "--output", str(EVAL_BM25_PATH),
    "--limit", "100",
    "--seed", "2026",
    "--ks", "1,3,5,10",
    "--bm25-top-k", "50",
    "--reranker-model", ""
]
ret = run(eval_bm25_cmd, cwd=REPO_DIR)
assert ret == 0, "Đo BM25 Retrieval thất bại!"

with open(EVAL_BM25_PATH, "r", encoding="utf-8") as f:
    bm25_report = json.load(f)

# Lấy metrics từ report["metrics"]
bm25_metrics = bm25_report.get("metrics", {}).get("bm25", {})
print("\n Kết quả BM25 Retrieval Metrics:")
print(json.dumps(bm25_metrics, indent=2, ensure_ascii=False))

assert "recall@5" in bm25_metrics or "recall_at_5" in bm25_metrics, "Report không chứa metric recall@5!"

TEST_STAGE_RESULTS["Tầng 3 (BM25 Index & Retrieval)"] = (ret == 0)
print("\n [TẦNG 3 PASS] BM25 Index & Retrieval độc lập hoạt động chính xác!")


---
## Tầng 4. Dense Index Độc Lập (Embedding Model Only)

1. **Sanity Encode Test**: Encode 3 câu tiếng Việt bằng `AITeamVN/Vietnamese_Embedding_v2`, kiểm tra shape `(3, dim)`, dtype `float32`, không có NaN/Inf, L2-norm xấp xỉ 1.0.
2. **Build Full Dense Vector Index**: Tạo vector FAISS / NumPy với checkpoint atomic, tự động xử lý rerun sạch sẽ với `--force` và `--resume`.
3. **Validate Dense Index against BM25**: Đảm bảo số lượng vector khớp chính xác số chunk trong SQLite.


In [ ]:
# 1. Sanity Encode Test với model thật
print("--- 4.1 KIỂM TRA MÔ HÌNH EMBEDDING THẬT TRÊN GPU/CPU ---")
from legalqa_baseline.dense import VietnameseEmbeddingModel
import numpy as np

print(f" Đang tải model: {EMBEDDING_MODEL_ID} trên device: {DEVICE}...")
t0 = time.time()
test_encoder = VietnameseEmbeddingModel(
    model_name_or_path=EMBEDDING_MODEL_ID,
    device=DEVICE,
)
print(f" Đã tải xong encoder sau {time.time()-t0:.2f}s.")

sample_queries = [
    "Trách nhiệm của tổ chức đấu thầu là gì?",
    "Quy định xử phạt vi phạm hành chính trong lĩnh vực môi trường.",
    "Điều kiện cấp giấy phép xây dựng nhà ở riêng lẻ.",
]

vectors = test_encoder.encode(sample_queries, max_length=256)
print(f" Vector Shape: {vectors.shape}")
print(f" Dtype: {vectors.dtype}")
print(f" Hữu hạn (No NaN/Inf): {np.isfinite(vectors).all()}")
l2_norms = np.linalg.norm(vectors, axis=1)
print(f" L2 Norms: {l2_norms}")

assert vectors.shape[0] == 3, f"Shape không đúng: {vectors.shape}"
assert vectors.dtype == np.float32, f"Dtype không phải float32: {vectors.dtype}"
assert np.isfinite(vectors).all(), "Vector chứa NaN hoặc Inf!"
assert np.allclose(l2_norms, 1.0, atol=1e-3), "Vector chưa được chuẩn hóa L2 (norm != 1.0)!"

# 2. Xây dựng Dense Index (Idempotent: kiểm tra nếu index đã có thì nạp, nếu chưa hoặc build lại thì dùng --force)
DENSE_INDEX_PATH = WORK_DIR / "legalqa_dense_test"
FORCE_REBUILD_DENSE = False

dense_meta_path = DENSE_INDEX_PATH.with_suffix(".meta.json")
dense_data_path = DENSE_INDEX_PATH.with_suffix(".npy")
dense_faiss_path = DENSE_INDEX_PATH.with_suffix(".faiss")

dense_already_built = dense_meta_path.exists() and (dense_data_path.exists() or dense_faiss_path.exists())

if dense_already_built and not FORCE_REBUILD_DENSE:
    print(f"\n [Dense Index] Đã tìm thấy Dense Index hoàn tất tại {DENSE_INDEX_PATH}, sử dụng lại.")
    ret = 0
else:
    print("\n--- 4.2 XÂY DỰNG DENSE VECTOR INDEX ---")
    build_dense_cmd = [
        sys.executable, "-m", "legalqa_baseline", "build-dense-index",
        "--contexts", str(CONTEXTS_PATH),
        "--dense-index", str(DENSE_INDEX_PATH),
        "--embedding-model", EMBEDDING_MODEL_ID,
        "--embedding-max-length", "2048",
        "--batch-size", "8",
        "--checkpoint-chunks", "4096",
        "--resume",
        "--force",
        "--device", DEVICE
    ]
    ret = run(build_dense_cmd, cwd=REPO_DIR)
    assert ret == 0, "Build Dense Index thất bại!"

# 3. Validate Dense Index
from legalqa_baseline.dense import DenseVectorIndex
from legalqa_baseline.storage import SearchIndex

with SearchIndex(str(DB_PATH)) as search_idx:
    bm25_meta = search_idx.metadata()
    dense_idx = DenseVectorIndex.load(str(DENSE_INDEX_PATH), expected_model_name=EMBEDDING_MODEL_ID)
    print(f" Số chunks trong Dense Index: {len(dense_idx.metadata):,}")
    dense_idx.validate_against_bm25(bm25_meta)
    print(" Dense Index tương thích hoàn toàn 100% với BM25 SQLite!")

TEST_STAGE_RESULTS["Tầng 4 (Dense Index & Embedding)"] = (ret == 0)
print("\n [TẦNG 4 PASS] Dense Index độc lập được xây dựng và xác thực thành công!")


---
## Tầng 5. So sánh BM25 – Dense – RRF Fusion (Chưa bật Reranker)

Đo lường và so sánh hiệu quả giữa 3 stage:
- `bm25`
- `dense`
- `rrf` (k=60 hợp nhất thứ hạng Top-50 từ BM25 và Dense)

**Mục tiêu quan sát:** `RRF Recall@5` phải lớn hơn hoặc bằng `BM25 Recall@5`.


In [ ]:
EVAL_RRF_PATH = WORK_DIR / "eval_dense_rrf_test.json"

print("--- 5. ĐO LƯỜNG BM25 VS DENSE VS RRF FUSION (100 MẪU) ---")
eval_rrf_cmd = [
    sys.executable, "-m", "legalqa_baseline", "evaluate-retrieval",
    "--train", str(TRAIN_PATH),
    "--db", str(DB_PATH),
    "--dense-index", str(DENSE_INDEX_PATH),
    "--output", str(EVAL_RRF_PATH),
    "--limit", "100",
    "--seed", "2026",
    "--ks", "1,3,5,10",
    "--embedding-model", EMBEDDING_MODEL_ID,
    "--reranker-model", "",
    "--device", DEVICE,
    "--bm25-top-k", "50",
    "--dense-top-k", "50",
    "--rrf-k", "60",
    "--rrf-top-k", "50"
]
ret = run(eval_rrf_cmd, cwd=REPO_DIR)
assert ret == 0, "Đo lường RRF Retrieval thất bại!"

with open(EVAL_RRF_PATH, "r", encoding="utf-8") as f:
    rrf_report = json.load(f)

# Đọc từ report["metrics"]
stages_metrics = rrf_report.get("metrics", {})
print("\n BẢNG SO SÁNH HIỆU QUẢ TRUY XUẤT 3 STAGES:")
header = f"{'Stage':<10} | {'Recall@1':<10} | {'Recall@3':<10} | {'Recall@5':<10} | {'MRR@5':<10} | {'Hit@5':<10}"
print("-" * len(header))
print(header)
print("-" * len(header))

for st_name in ["bm25", "dense", "rrf"]:
    m = stages_metrics.get(st_name, {})
    r1 = m.get("recall@1", m.get("recall_at_1", 0.0))
    r3 = m.get("recall@3", m.get("recall_at_3", 0.0))
    r5 = m.get("recall@5", m.get("recall_at_5", 0.0))
    mrr5 = m.get("mrr@5", m.get("mrr_at_5", 0.0))
    hit5 = m.get("hit@5", m.get("hit_at_5", 0.0))
    print(f"{st_name:<10} | {r1:<10.4f} | {r3:<10.4f} | {r5:<10.4f} | {mrr5:<10.4f} | {hit5:<10.4f}")

print("-" * len(header))
assert "rrf" in stages_metrics, "Báo cáo thiếu stage rrf!"
assert "dense" in stages_metrics, "Báo cáo thiếu stage dense!"

TEST_STAGE_RESULTS["Tầng 5 (BM25 vs Dense vs RRF)"] = (ret == 0)
print("\n [TẦNG 5 PASS] RRF Fusion kết hợp thành công BM25 và Dense Retrieval!")


---
## Tầng 6. Reranker Độc Lập & 4-Stage Comparison

Kích hoạt mô hình Cross-Encoder `AITeamVN/Vietnamese_Reranker` trên Top-50 candidate sau RRF để chọn ra Top-3 chunks chính xác nhất cho LLM.

Báo cáo sẽ có đầy đủ **4 stages**: `bm25`, `dense`, `rrf`, `reranker`.


In [ ]:
EVAL_RERANKER_PATH = WORK_DIR / "eval_reranker_test.json"

print("--- 6. ĐO LƯỜNG TOÀN DIỆN 4 STAGES KÈM VIETNAMESE RERANKER ---")
eval_rerank_cmd = [
    sys.executable, "-m", "legalqa_baseline", "evaluate-retrieval",
    "--train", str(TRAIN_PATH),
    "--db", str(DB_PATH),
    "--dense-index", str(DENSE_INDEX_PATH),
    "--output", str(EVAL_RERANKER_PATH),
    "--limit", "100",
    "--seed", "2026",
    "--ks", "1,3,5,8",
    "--embedding-model", EMBEDDING_MODEL_ID,
    "--reranker-model", RERANKER_MODEL_ID,
    "--device", DEVICE,
    "--bm25-top-k", "50",
    "--dense-top-k", "50",
    "--rrf-k", "60",
    "--rrf-top-k", "50",
    "--reranker-max-length", "2304"
]
ret = run(eval_rerank_cmd, cwd=REPO_DIR)
assert ret == 0, "Đo lường Reranker Retrieval thất bại!"

with open(EVAL_RERANKER_PATH, "r", encoding="utf-8") as f:
    rerank_report = json.load(f)

stages4_metrics = rerank_report.get("metrics", {})
print("\n BẢNG SO SÁNH HIỆU QUẢ TRUY XUẤT 4 STAGES:")
header4 = f"{'Stage':<10} | {'Recall@1':<10} | {'Recall@3':<10} | {'Recall@5':<10} | {'MRR@5':<10}"
print("-" * len(header4))
print(header4)
print("-" * len(header4))

for st_name in ["bm25", "dense", "rrf", "reranker"]:
    m = stages4_metrics.get(st_name, {})
    r1 = m.get("recall@1", m.get("recall_at_1", 0.0))
    r3 = m.get("recall@3", m.get("recall_at_3", 0.0))
    r5 = m.get("recall@5", m.get("recall_at_5", 0.0))
    mrr5 = m.get("mrr@5", m.get("mrr_at_5", 0.0))
    print(f"{st_name:<10} | {r1:<10.4f} | {r3:<10.4f} | {r5:<10.4f} | {mrr5:<10.4f}")

print("-" * len(header4))
assert "reranker" in stages4_metrics, "Báo cáo thiếu stage reranker!"

TEST_STAGE_RESULTS["Tầng 6 (Vietnamese Reranker)"] = (ret == 0)
print("\n [TẦNG 6 PASS] Reranker hoạt động và tinh chỉnh thứ hạng chính xác!")


---
## Tầng 7. Generator Độc Lập (Vi-Qwen2-1.5B-RAG Only)

Kiểm tra mô hình sinh câu trả lời độc lập với một đoạn văn bản luật và câu hỏi mẫu chuẩn.

Kiểm tra:
- Không lặp lại prompt.
- Không trả lời rỗng.
- Giữ nguyên số Điều, mức phạt, điều kiện pháp lý.
- Test `max_new_tokens = 512` vs `1024`.


In [ ]:
print("--- 7. KIỂM TRA GENERATOR LLM ĐỘC LẬP TRÊN GPU/CPU ---")
from legalqa_baseline.generator import ViQwenRAGGenerator

print(f" Đang khởi tạo Generator: {GENERATOR_MODEL_ID} trên device: {DEVICE}...")
t0 = time.time()
generator = ViQwenRAGGenerator(
    model_name_or_path=GENERATOR_MODEL_ID,
    device=DEVICE,
    max_new_tokens=1024,
    temperature=0.0,
)
print(f" Đã khởi tạo Generator sau {time.time()-t0:.2f}s.")

test_context = """Văn bản: Luật Giao thông đường bộ số 23/2008/QH12
Điều 9. Quy tắc chung
1. Người tham gia giao thông phải đi bên phải theo chiều đi của mình, đi đúng làn đường, phần đường quy định và phải chấp hành hệ thống báo hiệu đường bộ.
2. Xe ô tô có trang bị dây an toàn thì người lái xe và người ngồi hàng ghế phía trước trong xe ô tô phải thắt dây an toàn."""

test_question = "Người lái xe và người ngồi hàng ghế phía trước trong ô tô có bắt buộc phải thắt dây an toàn không?"

print("\n[Test 1] Sinh câu trả lời với context chuẩn:")
ans_1 = generator.generate(context=test_context, question=test_question)
print(f"Câu hỏi: {test_question}")
print(f"Đáp án sinh ra: {ans_1}")

assert len(ans_1.strip()) > 0, "Generator trả về đáp án rỗng!"
assert "dây an toàn" in ans_1.lower(), "Đáp án không chứa từ khóa chính từ ngữ cảnh!"

# Kiểm tra tính tất định khi temperature=0.0
print("\n[Test 2] Kiểm tra tính tất định (Deterministic check khi temp=0):")
ans_2 = generator.generate(context=test_context, question=test_question)
assert ans_1 == ans_2, "Temperature=0 nhưng Generator sinh 2 kết quả khác nhau!"
print(" Tính tất định đạt 100% (2 lần sinh giống hệt nhau).")

TEST_STAGE_RESULTS["Tầng 7 (Generator LLM)"] = True
print("\n [TẦNG 7 PASS] Generator LLM hoạt động chính xác và đáng tin cậy!")


---
## Tầng 8. Full RAG Pipeline trên 20 Mẫu Train

Chạy toàn bộ quy trình Hybrid RAG kết hợp từ đầu đến cuối trên 20 mẫu Train ngẫu nhiên.

> **Quan trọng:** Không truyền cờ `--allow-retrieval-fallback` để đảm bảo nếu Dense hoặc Reranker gặp bất kỳ lỗi nào, pipeline sẽ báo lỗi ngay (fail-fast) thay vì ngầm fallback về BM25.


In [ ]:
VAL_RAG_20_PATH = WORK_DIR / "validation_rag_20_test.json"

print("--- 8. CHẠY VALIDATION HYBRID RAG TRÊN 20 MẪU TRAIN ---")
val_cmd = [
    sys.executable, "-m", "legalqa_baseline", "validate",
    "--train", str(TRAIN_PATH),
    "--db", str(DB_PATH),
    "--dense-index", str(DENSE_INDEX_PATH),
    "--output", str(VAL_RAG_20_PATH),
    "--limit", "20",
    "--seed", "2026",
    "--modes", "rag",
    "--embedding-model", EMBEDDING_MODEL_ID,
    "--reranker-model", RERANKER_MODEL_ID,
    "--generator-model", GENERATOR_MODEL_ID,
    "--bm25-top-k", "50",
    "--dense-top-k", "50",
    "--rrf-top-k", "50",
    "--rerank-top-k", "3",
    "--context-top-k", "3",
    "--max-new-tokens", "1024",
    "--temperature", "0.0",
    "--device", DEVICE,
    "--official-metrics"
]
ret = run(val_cmd, cwd=REPO_DIR)
assert ret == 0, "Validation RAG thất bại!"

with open(VAL_RAG_20_PATH, "r", encoding="utf-8") as f:
    val_report = json.load(f)

print("\n KẾT QUẢ ĐIỂM SỐ CHÍNH THỨC TRÊN 20 CÂU:")
rag_scores = val_report.get("results", {}).get("rag", {})
for metric, val in rag_scores.items():
    if metric != "routes":
        print(f"   - {metric:<25}: {val}")

print(f"\n Phân bổ Routes: {rag_scores.get('routes', {})}")
assert rag_scores.get("samples", 0) == 20, "Số mẫu validation không đủ 20!"

TEST_STAGE_RESULTS["Tầng 8 (Full RAG 20 Mẫu)"] = (ret == 0)
print("\n [TẦNG 8 PASS] Toàn bộ pipeline Hybrid RAG chạy thành công trên 20 mẫu!")


---
## Tầng 9. Kiểm tra Định Dạng Submission & Cơ chế Resume

Tạo một tập dữ liệu test nhỏ (3 câu) để kiểm tra:
1. File submission có đúng chuẩn Codabench `{"id": {"answer": "..."}}`.
2. Không chứa các trường dư thừa (`route`, `evidence`, `confidence`).
3. Cơ chế `--resume` và lưu checkpoint atomic theo chu kỳ `CHECKPOINT_INTERVAL=1`.


In [ ]:
print("--- 9. KIỂM TRA TẠO SUBMISSION VÀ CƠ CHẾ RESUME ---")

# 1. Tạo file input mini
MINI_INPUT_PATH = WORK_DIR / "mini_test_input.json"
MINI_OUTPUT_PATH = WORK_DIR / "mini_submission.json"
MINI_CHECKPOINT_PATH = WORK_DIR / "mini_submission.checkpoint.json"

mini_data = {}
for i, (k, v) in enumerate(list(train_data.items())[:3], start=1):
    mini_data[f"test_{i:03d}"] = {"question": v["question"]}

with open(MINI_INPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(mini_data, f, ensure_ascii=False, indent=2)

print(f" Đã tạo file mini test: {len(mini_data)} câu.")

# 2. Chạy predict sinh submission
predict_cmd = [
    sys.executable, "-m", "legalqa_baseline", "predict",
    "--input", str(MINI_INPUT_PATH),
    "--db", str(DB_PATH),
    "--dense-index", str(DENSE_INDEX_PATH),
    "--output", str(MINI_OUTPUT_PATH),
    "--mode", "rag",
    "--embedding-model", EMBEDDING_MODEL_ID,
    "--reranker-model", RERANKER_MODEL_ID,
    "--generator-model", GENERATOR_MODEL_ID,
    "--context-top-k", "3",
    "--max-new-tokens", "512",
    "--checkpoint-interval", "1",
    "--device", DEVICE
]
ret = run(predict_cmd, cwd=REPO_DIR)
assert ret == 0, "Chạy Predict mini test thất bại!"

# 3. Kiểm tra schema submission
with open(MINI_OUTPUT_PATH, "r", encoding="utf-8") as f:
    sub_data = json.load(f)

print(f"\n Cấu trúc Submission:")
for k, v in sub_data.items():
    print(f"Sample '{k}': keys={list(v.keys())}, answer={v.get('answer', '')}")
    assert isinstance(v, dict), f"Entry '{k}' không phải là dict!"
    assert list(v.keys()) == ["answer"], f"Entry '{k}' chứa các key không hợp lệ: {list(v.keys())}!"
    assert isinstance(v["answer"], str) and len(v["answer"].strip()) > 0, f"Entry '{k}' có answer rỗng hoặc không phải string!"

# 4. Kiểm tra resume
print("\n Kiểm tra cơ chế Resume:")
predict_resume_cmd = predict_cmd + ["--resume"]
ret = run(predict_resume_cmd, cwd=REPO_DIR)
assert ret == 0, "Chạy Predict resume thất bại!"

TEST_STAGE_RESULTS["Tầng 9 (Submission & Resume)"] = (ret == 0)
print("\n [TẦNG 9 PASS] Định dạng Submission và cơ chế Resume hoạt động hoàn hảo!")


---
## Tầng 10. Kaggle Persistence & Tổng Kết Báo Cáo Thực Tế

Kiểm tra danh mục các artifacts được lưu trong `EXPORT_DIR` (trên Kaggle là `/kaggle/working`). Các file này sẽ được lưu giữ sau khi bấm **Save Version**, cho phép nạp lại ở lần chạy sau mà không cần build lại từ đầu.


In [ ]:
print("--- 10. KIỂM TRA ARTIFACTS PERSISTENCE CHO KAGGLE ---")

# Kiểm tra đường dẫn chính xác của artifacts (kể cả .meta.json và .npy / .faiss của Dense)
def check_artifact_existence(name: str, paths: list[Path]) -> tuple[str, str]:
    found = [p for p in paths if p.exists()]
    if found:
        total_mb = sum(p.stat().st_size for p in found if p.is_file()) / (1024 * 1024)
        detail = f"OK ({len(found)} files, {total_mb:.2f} MB)"
        return "EXISTS", detail
    return "MISSING", str(paths[0])

artifacts_to_verify = [
    ("BM25 SQLite Database", [DB_PATH]),
    ("BM25 Eval Report", [EVAL_BM25_PATH]),
    ("Dense Index Metadata", [DENSE_INDEX_PATH.with_suffix(".meta.json")]),
    ("Dense Vector Matrix (.npy/.faiss)", [DENSE_INDEX_PATH.with_suffix(".npy"), DENSE_INDEX_PATH.with_suffix(".faiss")]),
    ("RRF Eval Report", [EVAL_RRF_PATH]),
    ("Reranker Eval Report", [EVAL_RERANKER_PATH]),
    ("Validation 20 Samples Report", [VAL_RAG_20_PATH]),
    ("Mini Submission JSON", [MINI_OUTPUT_PATH]),
    ("Mini Checkpoint JSON", [MINI_CHECKPOINT_PATH]),
]

all_artifacts_ok = True
print(f"\n{'Artifact Name':<35} | {'Status':<10} | {'Detail'}")
print("-" * 80)
for name, path_list in artifacts_to_verify:
    status, detail = check_artifact_existence(name, path_list)
    if status == "MISSING":
        all_artifacts_ok = False
    print(f"{name:<35} | {status:<10} | {detail}")
print("-" * 80)

TEST_STAGE_RESULTS["Tầng 10 (Kaggle Persistence)"] = all_artifacts_ok

# In bảng tổng kết thực tế (Dynamic Evaluation Summary)
print("""
================================================================================
                    BẢNG TỔNG KẾT THỰC TẾ 10 TẦNG KIỂM THỬ
================================================================================""")
all_passed = True
for stage_name, passed in TEST_STAGE_RESULTS.items():
    badge = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f" {stage_name:<40}: {badge}")

print("================================================================================")
if all_passed and len(TEST_STAGE_RESULTS) == 10:
    print(" TOÀN BỘ 10 TẦNG ĐỀU PASS 100%! HỆ THỐNG SẴN SÀNG CHẠY FULL SUBMISSION TRÊN KAGGLE!")
else:
    print("  CẢNH BÁO: Có một số tầng chưa hoàn thành hoặc bị lỗi. Hãy kiểm tra các mục FAIL ở trên!")
